In [104]:
from sentence_transformers import SentenceTransformer
import numpy as np
import chromadb 
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [105]:
class EmbeddingModel:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()
    def _load_model(self):
        try:
            print(f"Loading Embedding Model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Loaded Successfully. Embedding Dimensions: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error Loading Model {self.model_name}: {e}")
            raise
    def generate_embeddings(self,texts:List[str]):
        if not self.model:
            raise ValueError("Model not Loaded")
        print(f"Generating Embedding for {len(texts)} texts...")
        embeddings = self.model.encode(texts,show_progress_bar=True)
        print(f"Generating embeddings with Shape : {embeddings.shape}")
        return embeddings
    
embedding_model = EmbeddingModel()
embedding_model

Loading Embedding Model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 368.19it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model Loaded Successfully. Embedding Dimensions: 384


### VectorStore

In [106]:
import os

In [107]:
class VectorStore:

    def __init__(self, collection_name: str="pdf_collections",
                 persist_directory: str="../data/vector_store"):

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):

        try:
            os.makedirs(self.persist_directory, exist_ok=True)

            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"documents": "PDF Documents Embeddings for RAG"}
            )

            print(f"Vector Store Initialized: {self.collection_name}")
            print(f"Existing documents: {self.collection.count()}")

        except Exception as e:
            print(f"Error Initializing Vector Store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):

        if len(documents) != len(embeddings):
            raise ValueError("Documents and embeddings count must match")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):

            ids.append(f"doc_{uuid.uuid4().hex[:8]}_{i}")

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)

            metadatas.append(metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(f"Added {len(documents)} documents")
            print(f"Total documents: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents: {e}")
            raise


vector_store = VectorStore()
vector_store

Vector Store Initialized: pdf_collections
Existing documents: 0


In [108]:
from langchain_community.document_loaders import PyPDFLoader

pdf = PyPDFLoader(
    "../data/pdf/information-systems.pdf",
    password= None,
    
)

In [109]:
documents = pdf.load()

In [110]:
print(documents)

[Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Information systems and technologies : textbook', 'author': 'O Pushkar, K S Sibilyev, О І Пушкар, К С Сібілєв, А И Пушкарь, К С Сибилев', 'subject': '', 'keywords': '01 Jan 2012, Information system, Software', 'source': '../data/pdf/information-systems.pdf', 'total_pages': 263, 'page': 0, 'page_label': '1'}, page_content='MINISTRY OF EDUCATION AND SCIENCE, \nYOUTH AND SPORTS OF UKRAINE \n \nKHARKIV NATIONAL UNIVERSITY OF ECONOMICS \n \n \n \n \n \n \n \nPushkar O. \nSibilyev K. \n \n \nINFORMATION SYSTEMS AND TECHNOLOGIES  \n \nTextbook \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nХарків. Вид. ХНЕУ, 2012'), Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Information systems and technologies : textbook', 'author': 'O Pushkar, K S Sibilyev, О І Пушкар, К С Сібілєв, А И Пушкарь, К С Сибилев', 'subject': '', 'keywords': '01 Jan 2012, Information system

In [111]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 512,
    chunk_overlap = 60,
    separators= ["\n\n","\n", " ", ""]
)

In [112]:
chuncking =text_splitter.split_documents(documents)

In [113]:
len(documents)

263

In [114]:
len(chuncking)

980

In [115]:
chunks = [chunck.page_content for chunck in chuncking]

In [116]:
emb = embedding_model.generate_embeddings(chunks)

Generating Embedding for 980 texts...


Batches:   0%|          | 0/31 [00:00<?, ?it/s]

Batches: 100%|██████████| 31/31 [00:50<00:00,  1.62s/it]

Generating embeddings with Shape : (980, 384)


In [117]:
vector_store.add_documents(chuncking,emb)

Added 980 documents
Total documents: 980


### Retriever Pipeline from VectorStore

In [118]:
class RAG_Retriever:
    def __init__(self, vector_store:VectorStore, embedding_model:EmbeddingModel):
        self.vector_store = vector_store
        self.embedding_model = embedding_model